# Deposits vs payment flows — are we seeing every action?

**PNC Treasury Management · DSI · Payment Knowledge Graph (PKG)**

Everything is at customer level (`customer_pwr_id`): every deposit account of a customer is already summed in
`pkg_dep_cust_month`, and every mdm id under the same power id is summed on the payment side. Read-only; tables
inline, figures saved to `OUT_DIR`.

**The test.** For each customer and month, the end-of-month balance change should equal the money we see moving:

balance change = payments in - payments out (internal + external) + own transfers in - out + interest paid - fees billed + what we cannot see

"What we cannot see" (the residual) is the size of the blind spot: cash deposits and withdrawals, debit card,
loan drawdowns and repayments, sweeps to investment accounts, account openings and closings, bank adjustments.

| § | Question |
|---|---|
| 0 | Inputs, window, and what the other candidate sources (cash view, deposit master, daily balances) contain |
| 1 | How many mdm ids sit under one power id |
| 2 | Who has what: deposits, payments, credit, TM products — and what the customers without deposits are |
| 3 | Customers with deposits but no payments: what they look like |
| 4 | Monthly reconciliation: how much of the balance change the payments explain, by group and by month |
| 5 | How much of the gap debit card explains (selected months) |
| 6 | Daily check on a sample: days the balance moves with no payment that day |
| 7 | Answer sheet |

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os, re, sys, time, math, warnings, threading, contextlib
import numpy as np
import pandas as pd

for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp),
               ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import DateType, TimestampType, NumericType

DB = "bdahd01p_dlcdi1_cdi_tm"
CONFIG = dict(
    DEP_TABLE   = f"{DB}.pkg_dep_cust_month",
    DAILY_TABLE = f"{DB}.pkg_dep_cust_daily",
    MET_TABLE   = f"{DB}.pkg_cust_metrics_monthly_v3",
    DIM_TABLE   = f"{DB}.pkg_cust_dim",
    TM_TABLE    = f"{DB}.pkg_tm_cust_month",
    PROD_TABLE  = f"{DB}.pkg_tm_cust_prod_month",
    EDGE_DAILY  = f"{DB}.pkg_edge_daily",
    PAY_TABLE   = "dsihd01p_dsi.neo4j_payments",
    DISCOVER    = ["clhhd01p_clh.cdm_tm_cor_trans_agg_v", "codhd01p_tm.deposit_master"],
    ID = "customer_pwr_id",
    REF_MONTH   = None,          # population map month; None = last month in deposits, metrics, TM and products
    CARD_MONTHS = None,          # None = last 2 months of the reconciliation window; [] = skip the card step
    RUN_DAILY   = True,
    DAILY_MONTH = None,          # None = last month in both the daily table and the daily edges
    DAILY_PCT   = 2,             # % of customers (by hash) in the daily check
    # daily table columns: None = detect from the schema (the chosen names are printed in §0)
    DAILY_DATE_COL = None, DAILY_BAL_COL = None,
    TOL_ABS = 100.0,             # a residual of $100 or less counts as explained
    TOL_REL = 0.05,              # ... or 5% or less of the observed activity
    PRODUCT_FLAGS = {            # TM product names (product table, column tm_product) used as cuts
        "branch services": ["BRANCH SERVICES DDA"],
        "sweep / ZBA": ["END OF DAY SWEEP", "BUSINESS SWEEP CHECKING", "ZBA", "CIB BALANCE AND FUNDING SHADOW"],
        "remote deposit": ["DEPOSIT ON SITE"],
    },
    OUT_DIR = "../eda/deposit_flow_recon",
    SHUFFLE_PARTITIONS = 400,
)
ID = CONFIG["ID"]

spark = (SparkSession.builder.appName("pkg_deposit_flow_recon")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.yarn.am.memory", "4g")
         .config("spark.yarn.am.memoryOverhead", "1g")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.kryoserializer.buffer.max", "128m")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
               "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
               "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False

def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def __init__(self):
        self.lines, self.tables, self.h = [], [], None

    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=80))

    def _draw(self):
        if self.h is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;color:#0b0b0b;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#8a8985;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()
TIMINGS = []


@contextlib.contextmanager
def step(name):
    """Timed step with a live line: elapsed minutes and Spark tasks of the running stage."""
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        TIMINGS.append((name, round(mins, 2)))
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


GATES = []


def gate(name, ok, detail, action=""):
    GATES.append(dict(gate=name, result="PASS" if ok else "FAIL", detail=detail, action=action))


def gates_frame():
    return pd.DataFrame(GATES, columns=["gate", "result", "detail", "action"])



OUT_DIR = CONFIG["OUT_DIR"]
os.makedirs(OUT_DIR, exist_ok=True)
FIG_INDEX, ANSWERS = [], []
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": "#b8b7b0", "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110,
})


def savefig(fig, name, caption):
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX.append(dict(file=f"{name}.png", caption=caption))


def ans(q, a):
    ANSWERS.append(dict(question=q, answer=a))


def table_months(tbl, col="month"):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().iloc[:, 0]
        m = sorted({x.split(f"{col}=")[1].split("/")[0] for x in p if f"{col}=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(str(r[0]) for r in spark.table(tbl).select(col).distinct().collect())


def mi_col(c):
    c = F.col(c) if isinstance(c, str) else c
    return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int")).cast("int")


def mi2m(i):
    i = int(i)
    return f"{(i - 1) // 12:04d}-{(i - 1) % 12 + 1:02d}"


def m2mi(m):
    return int(m[:4]) * 12 + int(m[5:7])


def pct_tbl(df, cols):
    for c in cols:
        df[c] = df[c].map(lambda v: "" if pd.isna(v) else f"{100 * v:.1f}%")
    return df


BANDS = [(-np.inf, 1_000, "under 1K"), (1_000, 10_000, "1K-10K"), (10_000, 100_000, "10K-100K"),
         (100_000, 1_000_000, "100K-1M"), (1_000_000, np.inf, "1M and over")]


def band(col):
    e = F.lit(BANDS[-1][2])
    for lo, hi, lab in reversed(BANDS[:-1]):
        e = F.when(F.col(col) < hi, F.lit(lab)).otherwise(e)
    return e

## §0 · Inputs, window and other candidate sources

In [ ]:
CELL.start("§0 · Inputs")
SRC = {}
with step("schemas and months"):
    _rows = []
    for k in ["DEP_TABLE", "MET_TABLE", "DIM_TABLE", "TM_TABLE", "PROD_TABLE", "DAILY_TABLE", "EDGE_DAILY", "PAY_TABLE"]:
        t = CONFIG[k]
        try:
            sdf = spark.table(t)
            SRC[k] = sdf
            ms = table_months(t) if "month" in sdf.columns else []
            _rows.append(dict(input=k, table=t, columns=len(sdf.columns),
                              months=f"{ms[0]} .. {ms[-1]} ({len(ms)})" if ms else ""))
        except Exception as ex:
            _rows.append(dict(input=k, table=t, columns=0, months=f"not readable: {type(ex).__name__}"))
    _disc = []
    for t in CONFIG["DISCOVER"]:
        try:
            cols = spark.table(t).dtypes
            hit = [f"{c} ({ty})" for c, ty in cols
                   if re.search(r"cash|curr|coin|atm|branch|teller|wdl|withdr|deposit|dep_|chnl|channel|card|check|chk|"
                                r"lockbox|rdc|sweep|loan|fee|_amt|_cnt|acct|pwr|month|_dt|date|key", c, re.I)]
            _disc.append(dict(table=t, columns=len(cols), relevant_columns=", ".join(hit),
                              all_columns=", ".join(c for c, _ in cols)))
        except Exception as ex:
            _disc.append(dict(table=t, columns=0, relevant_columns=f"not readable: {type(ex).__name__}", all_columns=""))
    _disc = pd.DataFrame(_disc)
    _disc.to_csv(os.path.join(OUT_DIR, "other_sources_columns.csv"), index=False)

for need in ["DEP_TABLE", "MET_TABLE", "DIM_TABLE"]:
    if need not in SRC:
        raise ValueError(f"{CONFIG[need]} is required and not readable.")
dep, met, dim = SRC["DEP_TABLE"], SRC["MET_TABLE"], SRC["DIM_TABLE"]
tm, prod = SRC.get("TM_TABLE"), SRC.get("PROD_TABLE")
DEP_M, MET_M = table_months(CONFIG["DEP_TABLE"]), table_months(CONFIG["MET_TABLE"])
REC_MONTHS = [m for m in MET_M if m in DEP_M and mi2m(m2mi(m) - 1) in DEP_M]
if not REC_MONTHS:
    raise ValueError("No month has deposits for the month and the month before plus customer metrics.")
_common = set(REC_MONTHS)
if tm is not None:
    _common &= set(table_months(CONFIG["TM_TABLE"]))
if prod is not None:
    _common &= set(table_months(CONFIG["PROD_TABLE"]))
REF = CONFIG["REF_MONTH"] or max(_common)
CARD_MONTHS = REC_MONTHS[-2:] if CONFIG["CARD_MONTHS"] is None else CONFIG["CARD_MONTHS"]

# daily table: detect the date and balance columns
DAILY_OK, _dmsg = False, ""
if CONFIG["RUN_DAILY"] and "DAILY_TABLE" in SRC and "EDGE_DAILY" in SRC:
    dd = SRC["DAILY_TABLE"]
    _dt = CONFIG["DAILY_DATE_COL"] or next((f.name for f in dd.schema.fields if isinstance(f.dataType, (DateType, TimestampType))), None) \
        or next((c for c in dd.columns if re.search(r"(^|_)(date|dt|day)$", c)), None)
    if CONFIG["DAILY_BAL_COL"]:
        _bal = CONFIG["DAILY_BAL_COL"]
    else:
        _cands = [f.name for f in dd.schema.fields if isinstance(f.dataType, NumericType) and re.search(r"bal", f.name)
                  and not re.search(r"avg|mean|min|max|std|first|_m$|mtd|ytd|dda|mmda|cd_|retail|sweep|other", f.name)]
        _bal = _cands[0] if len(_cands) == 1 else None
        if len(_cands) > 1:
            _dmsg = "several balance columns: " + ", ".join(_cands) + " (set DAILY_BAL_COL)"
    if _dt and _bal and ID in dd.columns:
        DAILY_OK, DAILY_COLS = True, (_dt, _bal)
        _dmsg = f"date column {_dt}, balance column {_bal}"
    elif not _dmsg:
        _dmsg = f"could not find {'the date column' if not _dt else ''} {'the balance column' if not _bal else ''} {'' if ID in dd.columns else ID}; set DAILY_DATE_COL / DAILY_BAL_COL"
    _dsch = ", ".join(f"{c} ({t})" for c, t in dd.dtypes)
else:
    _dmsg, _dsch = "skipped (RUN_DAILY off or table not readable)", ""
_win = pd.DataFrame([dict(item="reconciliation months", value=f"{REC_MONTHS[0]} .. {REC_MONTHS[-1]} ({len(REC_MONTHS)})"),
                     dict(item="population map month", value=REF),
                     dict(item="debit card months", value=", ".join(CARD_MONTHS) or "skipped"),
                     dict(item="daily check", value=_dmsg),
                     dict(item="daily table columns", value=_dsch)])
CELL.show(("Inputs", pd.DataFrame(_rows)), ("Window", _win),
          ("Other sources: columns that may hold cash, card, loans, sweeps (all columns in other_sources_columns.csv)",
           _disc[["table", "columns", "relevant_columns"]]))

## §1 · Power ids and mdm ids

The deposit table is one row per power id (all its accounts summed). Payments are recorded per mdm id. How many mdm
ids does a power id have, and does it matter for the customers that have deposits?

In [ ]:
CELL.start("§1 · Power ids and mdm ids")
with step("mdm ids per power id"):
    _map = (dim.where(F.col(ID).isNotNull()).select(F.col("mdm_id").cast("string").alias("mdm_id"),
                                                   F.col(ID).cast("string").alias(ID)).distinct())
    _per = _map.groupBy(ID).agg(F.count(F.lit(1)).alias("n_mdm"))
    _depids = dep.where(F.col("month") == REF).select(F.col(ID).cast("string").alias(ID)).distinct().withColumn("has_dep", F.lit(1))
    _c = (_per.join(_depids, ID, "full")
          .withColumn("n_mdm", F.coalesce("n_mdm", F.lit(0)))
          .withColumn("mdm_band", F.when(F.col("n_mdm") == 0, "0 (not in pkg_cust_dim)").when(F.col("n_mdm") == 1, "1")
                      .when(F.col("n_mdm") == 2, "2").otherwise("3 or more"))
          .groupBy("mdm_band").agg(F.count(F.lit(1)).alias("power_ids"),
                                   F.sum(F.coalesce("has_dep", F.lit(0))).alias(f"with_deposits_{REF}"))
          .orderBy("mdm_band").toPandas())
CELL.show(("mdm ids per power id (all of them are summed on the payment side)", _c))
_one = _c.loc[_c.mdm_band == "1", f"with_deposits_{REF}"].sum() / max(_c[f"with_deposits_{REF}"].sum(), 1)
ans("Deposit customers with exactly one mdm id", f"{_one:.1%}")

## §2 · Who has what

Every power id seen in any of the tables in the reference month, sorted into one segment:

| Segment | Rule |
|---|---|
| 1 deposits + payments | deposit row and at least one payment in or out |
| 2 deposits, no payments | deposit row, no payment that month |
| 3 payments, no deposits | payments through PNC accounts but no deposit row: a gap in the deposit data |
| 4 credit only | no deposits, no payments, direct hard exposure (DHE) above zero |
| 5 TM products only | none of the above, but in account analysis or holding TM products |
| 6 in the TM table only | none of the above (former or dormant customers kept in the table) |

Each row shows the share of organisations, median balance, account analysis, credit, the most held TM products.
Saved: f01.

In [ ]:
CELL.start(f"§2 · Who has what ({REF})")
with step("population map"):
    sid = lambda df: df.withColumn(ID, F.col(ID).cast("string"))
    _d = sid(dep.where(F.col("month") == REF)).groupBy(ID).agg(
        F.lit(1).alias("has_dep"), F.max(F.col("is_org").cast("int")).alias("dep_org"),
        F.sum("dep_bal_eom").alias("bal_eom"), F.sum("n_acct").alias("n_acct"))
    _p = (sid(met.where((F.col("month") == REF) & F.col(ID).isNotNull())).groupBy(ID)
          .agg(F.sum("amt_all_in").alias("pay_in"), F.sum("amt_all_out").alias("pay_out"),
               F.sum(F.coalesce(F.col("amt_self_in"), F.lit(0.0)) + F.coalesce(F.col("amt_self_out"), F.lit(0.0))).alias("pay_self"))
          .withColumn("has_pay", ((F.col("pay_in") + F.col("pay_out") + F.col("pay_self")) > 0).cast("int")))
    _u = _d.join(_p, ID, "full")
    if tm is not None:
        _t = sid(tm.where(F.col("month") == REF)).groupBy(ID).agg(
            F.lit(1).alias("in_tm"), F.max("in_ofs").alias("in_ofs"), F.max("cust_gone").alias("cust_gone"),
            F.max("dhe").alias("dhe"), F.max(F.when(F.col("credit_exposure_type") == "BL", 1).otherwise(0)).alias("bl"))
        _u = _u.join(_t, ID, "full")
    if prod is not None:
        _pr = sid(prod.where(F.col("month") == REF)).groupBy(ID).agg(F.countDistinct("tm_product").alias("n_prod"))
        _u = _u.join(_pr, ID, "full")
    _dm = (sid(dim.where(F.col(ID).isNotNull())).groupBy(ID).agg(F.max(F.when(F.col("party_type") == "O", 1).otherwise(0)).alias("dim_org"),
                            F.max(F.col("is_tm").cast("int")).alias("dim_tm")))
    _u = _u.join(_dm, ID, "left")
    for c in ["has_dep", "has_pay", "in_tm", "in_ofs", "cust_gone", "bl", "n_prod", "dim_org", "dep_org"]:
        if c not in _u.columns:
            _u = _u.withColumn(c, F.lit(0))
    for c in ["dhe"]:
        if c not in _u.columns:
            _u = _u.withColumn(c, F.lit(0.0))
    _u = _u.na.fill(0, ["has_dep", "has_pay", "in_tm", "in_ofs", "cust_gone", "bl", "n_prod", "dim_org", "dep_org", "dhe"])
    _u = _u.withColumn("segment",
                       F.when((F.col("has_dep") == 1) & (F.col("has_pay") == 1), "1 deposits + payments")
                       .when(F.col("has_dep") == 1, "2 deposits, no payments")
                       .when(F.col("has_pay") == 1, "3 payments, no deposits")
                       .when(F.col("dhe") > 0, "4 credit only")
                       .when((F.col("in_ofs") == 1) | (F.col("n_prod") > 0), "5 TM products only")
                       .otherwise("6 in the TM table only"))
    U = _u.persist()
    _seg = (U.groupBy("segment").agg(
        F.count(F.lit(1)).alias("power_ids"),
        F.avg(F.greatest("dim_org", "dep_org").cast("double")).alias("organisations"),
        F.expr("percentile_approx(bal_eom, 0.5, 1000)").alias("median_balance"),
        F.avg("in_ofs").alias("in_account_analysis"), F.avg((F.col("dhe") > 0).cast("double")).alias("with_dhe"),
        F.avg((F.col("n_prod") > 0).cast("double")).alias("with_tm_products"),
        F.avg("cust_gone").alias("flagged_gone"), F.avg("in_tm").alias("in_tm_table"))
        .orderBy("segment").toPandas())
    _seg.insert(2, "share", _seg.power_ids / _seg.power_ids.sum())
    _top = pd.DataFrame()
    if prod is not None:
        _pp = (sid(prod.where(F.col("month") == REF)).select(ID, "tm_product").distinct()
               .join(U.select(ID, "segment"), ID).groupBy("segment", "tm_product").agg(F.count(F.lit(1)).alias("n"))
               .withColumn("rk", F.row_number().over(W.partitionBy("segment").orderBy(F.desc("n")))).where(F.col("rk") <= 6)
               .toPandas())
        _segn = _seg.set_index("segment").power_ids
        _top = (_pp.sort_values(["segment", "rk"]).groupby("segment")
                .apply(lambda g: ", ".join(f"{p} ({n / _segn[g.name]:.0%})" for p, n in zip(g.tm_product, g.n)))
                .rename("most_held_tm_products").reset_index())
_seg_show = pct_tbl(_seg.copy(), ["share", "organisations", "in_account_analysis", "with_dhe", "with_tm_products",
                                  "flagged_gone", "in_tm_table"])
fig, ax = plt.subplots(figsize=(9, 3.8))
_s = _seg.sort_values("segment", ascending=False)
_segc = {"1": C1, "2": "#7aa6e0", "3": C2, "4": C3, "5": C4, "6": "#8a8985"}
ax.barh(_s.segment, _s.power_ids, color=[_segc.get(s[:1], C1) for s in _s.segment], height=0.65)
for y, v in enumerate(_s.power_ids):
    ax.text(v, y, f"  {v:,.0f}", va="center", fontsize=9, color=INK)
ax.set_title(f"Power ids by segment, {REF}"); ax.set_xlabel("power ids"); ax.grid(axis="y", visible=False)
savefig(fig, "f01_segments", f"Power ids by segment in {REF}")
CELL.show(("Segments", _seg_show), ("Most held TM products per segment (share of the segment)", _top))
for r in _seg.itertuples():
    ans(f"{REF}: {r.segment}", f"{r.power_ids:,} power ids ({r.share:.1%})")

## §3 · Deposits but no payments

Segment 2 against segment 1, same month: balance size, where the money sits (DDA, retail, MMDA, CD, sweep, other),
whether the balance moved at all (no change / interest only / moved without a payment), and when the customer last
made or received a payment in PKG.

In [ ]:
CELL.start(f"§3 · Deposits but no payments ({REF})")
_prev = mi2m(m2mi(REF) - 1)
with step("profile of segments 1 and 2"):
    _parts = ["dda", "retail", "mmda", "cd", "sweep_on", "sweep_off", "other"]
    _parts = [p for p in _parts if f"bal_{p}_eom" in dep.columns]
    _dd = sid(dep.where(F.col("month").isin(REF, _prev))).groupBy(ID).agg(
        *[F.sum(F.when(F.col("month") == REF, F.col(c))).alias(c) for c in
          ["dep_bal_eom", "interest_paid_month", "n_acct_new", "n_acct_closed", "n_acct_dormant_new"] + [f"bal_{p}_eom" for p in _parts]
          if c in dep.columns],
        F.sum(F.when(F.col("month") == _prev, F.col("dep_bal_eom"))).alias("prev_eom"))
    _hist = (sid(met.where(F.col(ID).isNotNull() & (F.col("month") <= REF))).groupBy(ID)
             .agg(F.max(F.when((F.col("amt_all_in") + F.col("amt_all_out")) > 0, F.col("month"))).alias("last_pay_month")))
    _s12 = (U.where(F.col("segment").isin("1 deposits + payments", "2 deposits, no payments")).select(ID, "segment", "dim_org", "dep_org")
            .join(_dd, ID, "left").join(_hist, ID, "left")
            .withColumn("band", band("dep_bal_eom"))
            .withColumn("d", F.col("dep_bal_eom") - F.col("prev_eom"))
            .withColumn("movement", F.when(F.col("prev_eom").isNull(), "new this month")
                        .when(F.abs(F.col("d")) <= 1, "no change")
                        .when(F.abs(F.col("d") - F.coalesce(F.col("interest_paid_month"), F.lit(0.0))) <= 1, "interest only")
                        .otherwise("moved"))
            .withColumn("last_pay", F.when(F.col("last_pay_month").isNull(), "never in PKG (since 2024-01)")
                        .when(mi_col(F.lit(REF)) - mi_col("last_pay_month") <= 0, "this month")
                        .when(mi_col(F.lit(REF)) - mi_col("last_pay_month") <= 3, "1-3 months ago")
                        .when(mi_col(F.lit(REF)) - mi_col("last_pay_month") <= 12, "4-12 months ago")
                        .otherwise("over 12 months ago")))
    S12 = _s12.persist()

    def _dist(col, order=None):
        t = S12.groupBy("segment", col).agg(F.count(F.lit(1)).alias("n")).toPandas()
        t["share"] = t.n / t.groupby("segment").n.transform("sum")
        w = t.pivot(index=col, columns="segment", values="share").fillna(0)
        if order:
            w = w.reindex([o for o in order if o in w.index])
        return pct_tbl(w.reset_index(), [c for c in w.columns])

    _band = _dist("band", [b[2] for b in BANDS])
    _mov = _dist("movement", ["no change", "interest only", "moved", "new this month"])
    _lp = _dist("last_pay", ["this month", "1-3 months ago", "4-12 months ago", "over 12 months ago", "never in PKG (since 2024-01)"])
    _mix = (S12.groupBy("segment").agg(F.sum("dep_bal_eom").alias("total"),
                                       *[F.sum(f"bal_{p}_eom").alias(p) for p in _parts]).toPandas().set_index("segment"))
    _mix = _mix[_parts].div(_mix["total"], axis=0).T.reset_index().rename(columns={"index": "where the balance sits"})
    _mix = pct_tbl(_mix, [c for c in _mix.columns if c != "where the balance sits"])
    _org = S12.groupBy("segment").agg(F.avg(F.greatest("dim_org", "dep_org").cast("double")).alias("organisations"),
                                      F.sum(F.when(F.col("n_acct_dormant_new") > 0, 1).otherwise(0)).alias("new_dormant_accounts"),
                                      F.count(F.lit(1)).alias("power_ids")).toPandas()
CELL.show(("Balance size (share of the segment)", _band), ("Where the balance sits (share of segment balance)", _mix),
          (f"Did the balance move from {_prev} to {REF}?", _mov), ("Last month with a payment in PKG", _lp),
          ("Organisations and dormancy", pct_tbl(_org, ["organisations"])))

## §4 · Monthly reconciliation

For every power id with deposits in a month and the month before:

- **balance change** = end-of-month balance - previous end-of-month balance
- **seen** = payments in - payments out (internal + external) + own transfers in - out + interest paid - TM fees billed
- **residual** = balance change - seen; **activity** = payments in + out + |residual|
- **unexplained share** = |residual| / activity. Explained: residual within 100 dollars or 5% of activity.

Customer-months where an account opened or closed are kept but shown as their own group (money can arrive or leave
with the account). Saved: f02 (unexplained share by group), f03 (book level by month), f04 (seen against balance change).

In [ ]:
CELL.start("§4 · Monthly reconciliation")
with step("balance changes, flows, fees, product flags"):
    _dep = (sid(dep.where(F.col("month") >= mi2m(m2mi(REC_MONTHS[0]) - 1))).groupBy(ID, "month").agg(
        F.sum("dep_bal_eom").alias("eom"), F.sum(F.coalesce(F.col("interest_paid_month"), F.lit(0.0))).alias("interest"),
        F.sum(F.coalesce(F.col("n_acct_new"), F.lit(0))).alias("acct_new"),
        F.sum(F.coalesce(F.col("n_acct_closed"), F.lit(0))).alias("acct_closed"),
        F.max(F.col("is_org").cast("int")).alias("is_org"))
        .withColumn("mi", mi_col("month")))
    _w = W.partitionBy(ID).orderBy("mi")
    _dep = (_dep.withColumn("prev_eom", F.lag("eom").over(_w)).withColumn("prev_mi", F.lag("mi").over(_w))
            .where(F.col("month").isin(REC_MONTHS) & (F.col("prev_mi") == F.col("mi") - 1)))
    _flow = (sid(met.where(F.col(ID).isNotNull() & F.col("month").isin(REC_MONTHS))).groupBy(ID, "month").agg(
        F.sum("amt_all_in").alias("pay_in"), F.sum("amt_all_out").alias("pay_out"),
        F.sum(F.coalesce(F.col("amt_self_in"), F.lit(0.0))).alias("self_in"),
        F.sum(F.coalesce(F.col("amt_self_out"), F.lit(0.0))).alias("self_out")))
    R = _dep.join(_flow, [ID, "month"], "left")
    if tm is not None and "billed" in tm.columns:
        R = R.join(sid(tm.where(F.col("month").isin(REC_MONTHS))).groupBy(ID, "month").agg(F.sum("billed").alias("fees")),
                   [ID, "month"], "left")
    else:
        R = R.withColumn("fees", F.lit(0.0))
    if prod is not None:
        _flags = sid(prod.where(F.col("month").isin(REC_MONTHS))).groupBy(ID, "month").agg(
            *[F.max(F.col("tm_product").isin(v).cast("int")).alias(f"p_{i}") for i, v in enumerate(CONFIG["PRODUCT_FLAGS"].values())])
        R = R.join(_flags, [ID, "month"], "left")
    else:
        for i in range(len(CONFIG["PRODUCT_FLAGS"])):
            R = R.withColumn(f"p_{i}", F.lit(0))
    R = (R.na.fill(0.0, ["pay_in", "pay_out", "self_in", "self_out", "fees"])
         .na.fill(0, [f"p_{i}" for i in range(len(CONFIG["PRODUCT_FLAGS"]))])
         .withColumn("d", F.col("eom") - F.col("prev_eom"))
         .withColumn("seen", F.col("pay_in") - F.col("pay_out") + F.col("self_in") - F.col("self_out")
                     + F.col("interest") - F.col("fees"))
         .withColumn("resid", F.col("d") - F.col("seen"))
         .withColumn("activity", F.col("pay_in") + F.col("pay_out") + F.abs("resid"))
         .withColumn("unexpl", F.when(F.col("activity") > 0, F.abs("resid") / F.col("activity")))
         .withColumn("cls", F.when((F.abs("d") <= 1) & (F.col("pay_in") + F.col("pay_out") <= 0), "no movement")
                     .when((F.abs("resid") <= CONFIG["TOL_ABS"]) | (F.col("unexpl") <= CONFIG["TOL_REL"]), "explained")
                     .when(F.col("unexpl") <= 0.5, "partly explained")
                     .otherwise("mostly unexplained"))
         .withColumn("acct_change", ((F.col("acct_new") > 0) | (F.col("acct_closed") > 0)).cast("int"))
         .withColumn("payments", F.when(F.col("pay_in") + F.col("pay_out") > 0, "with payments").otherwise("no payments"))
         .withColumn("band", band("prev_eom")))
    R = R.persist()

    CLS = ["no movement", "explained", "partly explained", "mostly unexplained"]

    def _by(expr, label):
        t = (R.groupBy(expr.alias("group")).agg(F.count(F.lit(1)).alias("customer_months"),
                                *[F.avg((F.col("cls") == c).cast("double")).alias(c) for c in CLS],
                                (F.sum(F.abs("resid")) / F.sum("activity")).alias("dollar_unexplained_share"),
                                (F.sum(F.when(F.col("resid") > 0, F.col("resid")).otherwise(0.0)) / F.sum(F.abs("resid"))).alias("residual_that_is_money_in"))
             .orderBy("group").toPandas())
        t.insert(0, "cut", label)
        return t

    _ov = _by(F.lit("all"), "all")
    _cuts = pd.concat([_ov,
                       _by(F.when(F.col("is_org") == 1, "organisation").otherwise("person / other"), "party"),
                       _by(F.col("payments"), "payments that month"),
                       _by(F.when(F.col("acct_change") == 1, "account opened or closed").otherwise("no account change"), "accounts"),
                       _by(F.col("band"), "previous balance")]
                      + [_by(F.when(F.col(f"p_{i}") == 1, "holds it").otherwise("does not"), f"product: {k}")
                         for i, k in enumerate(CONFIG["PRODUCT_FLAGS"])], ignore_index=True)
    _mon = (R.groupBy("month").agg(F.count(F.lit(1)).alias("customer_months"),
                                   F.sum("pay_in").alias("pay_in"), F.sum("pay_out").alias("pay_out"),
                                   F.sum("d").alias("balance_change"), F.sum("seen").alias("seen_net"),
                                   F.sum(F.when(F.col("resid") > 0, F.col("resid")).otherwise(0.0)).alias("unseen_in"),
                                   F.sum(F.when(F.col("resid") < 0, -F.col("resid")).otherwise(0.0)).alias("unseen_out"),
                                   F.sum("activity").alias("activity"),
                                   F.avg((F.col("cls") == "explained").cast("double")).alias("explained"))
            .orderBy("month").toPandas())
    _mon["dollar_unexplained_share"] = (_mon.unseen_in + _mon.unseen_out) / _mon.activity
    _smp = (R.where(F.col("cls") != "no movement").sample(False, min(1.0, 150_000 / max(R.count(), 1)), seed=5)
            .select("d", "seen", "is_org").toPandas())

_cuts_show = pct_tbl(_cuts.copy(), CLS + ["dollar_unexplained_share", "residual_that_is_money_in"])
_mon_show = pct_tbl(_mon.copy(), ["explained", "dollar_unexplained_share"])
# f02: stacked share of classes by group
_g = _cuts[_cuts.cut != "all"].copy()
_g["label"] = _g.cut + ": " + _g.group.astype(str)
fig, ax = plt.subplots(figsize=(10, 0.36 * len(_g) + 1.6))
left = np.zeros(len(_g))
for c, col in zip(CLS, ["#c9c8c2", C3, C4, C2]):
    ax.barh(_g.label, _g[c], left=left, color=col, label=c, height=0.7, edgecolor=SURF, linewidth=1)
    left += _g[c].to_numpy()
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Customer-months: is the balance change explained by the payments we see?", pad=26)
ax.legend(ncol=4, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9)
savefig(fig, "f02_explained_by_group", "Share of customer-months explained / partly / mostly unexplained, by group")
# f03: book level by month
fig, ax = plt.subplots(figsize=(11, 4.2))
x = np.arange(len(_mon))
ax.bar(x, _mon.pay_in / 1e9, color=C1, width=0.4, label="payments in (seen)")
ax.bar(x + 0.4, _mon.unseen_in / 1e9, color=C2, width=0.4, label="unseen money in")
ax.bar(x, -_mon.pay_out / 1e9, color="#7aa6e0", width=0.4, label="payments out (seen)")
ax.bar(x + 0.4, -_mon.unseen_out / 1e9, color="#f0a37f", width=0.4, label="unseen money out")
ax.axhline(0, color=INK2, linewidth=1)
ax.set_xticks(x + 0.2); ax.set_xticklabels(_mon.month, rotation=60, fontsize=8)
ax.set_ylabel("USD billions per month"); ax.grid(axis="x", visible=False)
ax.set_title("Deposit customers: seen payments against the money we cannot see", pad=26)
ax.legend(ncol=4, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9)
savefig(fig, "f03_seen_vs_unseen_by_month", "Seen payments in/out against unseen money in/out, all deposit customers, by month")
# f04: seen net flow against balance change (symlog)
if len(_smp):
    fig, ax = plt.subplots(figsize=(6.4, 6))
    sl = lambda v: np.sign(v) * np.log10(1 + np.abs(v))
    hb = ax.hexbin(sl(_smp.seen), sl(_smp.d), gridsize=70, cmap="Blues", mincnt=1, bins="log")
    lim = [min(sl(_smp.seen).min(), sl(_smp.d).min()), max(sl(_smp.seen).max(), sl(_smp.d).max())]
    ax.plot(lim, lim, color=INK2, linewidth=1, linestyle=":")
    ax.set_xlabel("seen net flow (signed log10 USD)"); ax.set_ylabel("balance change (signed log10 USD)")
    ax.set_title("Each customer-month: seen net flow vs balance change"); ax.grid(False)
    fig.colorbar(hb, ax=ax, fraction=0.04, pad=0.01).set_label("customer-months")
    savefig(fig, "f04_seen_vs_balance_change", "Seen net flow against balance change per customer-month (dotted = fully explained)")
CELL.show(("Customer-months by group (share of each class)", _cuts_show), ("Book level by month", _mon_show))
_a = _ov.iloc[0]
ans("Customer-months explained by seen payments (all months)", f"{_a['explained']:.1%}; no movement {_a['no movement']:.1%}; "
    f"mostly unexplained {_a['mostly unexplained']:.1%}")
ans("Dollars of activity we cannot see (all months)", f"{_a['dollar_unexplained_share']:.1%}; "
    f"{_a['residual_that_is_money_in']:.0%} of it is money coming in")

## §5 · Debit card

Debit card (categories 8B out, 8C in) was left out of the PKG edges and metrics. For the card months, add it back
from the payment table and see how much of the residual it removes.

In [ ]:
if CARD_MONTHS and "PAY_TABLE" in SRC:
    CELL.start("§5 · Debit card")
    with step(f"debit card from the payment table ({', '.join(CARD_MONTHS)})"):
        pay = SRC["PAY_TABLE"]
        _lo, _hi = CARD_MONTHS[0] + "-01", CARD_MONTHS[-1] + "-31"
        _cd = pay.where(F.col("category").isin("8B", "8C") & F.col("trans_dt").between(_lo, _hi))
        _card = (_cd.select(F.substring("trans_dt", 1, 7).alias("month"),
                            F.when(F.col("category") == "8B", F.col("mdm_id_pays")).otherwise(F.col("mdm_id_receives")).cast("string").alias("mdm_id"),
                            F.when(F.col("category") == "8B", F.abs("trans_amt")).otherwise(F.lit(0.0)).alias("card_out"),
                            F.when(F.col("category") == "8C", F.abs("trans_amt")).otherwise(F.lit(0.0)).alias("card_in"))
                 .where(F.col("mdm_id").isNotNull() & F.col("month").isin(CARD_MONTHS))
                 .join(_map, "mdm_id")
                 .groupBy(ID, "month").agg(F.sum("card_in").alias("card_in"), F.sum("card_out").alias("card_out")))
        Rc = (R.where(F.col("month").isin(CARD_MONTHS)).join(_card, [ID, "month"], "left").na.fill(0.0, ["card_in", "card_out"])
              .withColumn("resid2", F.col("resid") - (F.col("card_in") - F.col("card_out")))
              .withColumn("activity2", F.col("pay_in") + F.col("pay_out") + F.col("card_in") + F.col("card_out") + F.abs("resid2"))
              .withColumn("expl1", (F.abs("resid") <= CONFIG["TOL_ABS"]) | (F.abs("resid") <= CONFIG["TOL_REL"] * F.col("activity")))
              .withColumn("expl2", (F.abs("resid2") <= CONFIG["TOL_ABS"]) | (F.abs("resid2") <= CONFIG["TOL_REL"] * F.col("activity2"))))
        _cr = (Rc.groupBy(F.when(F.col("is_org") == 1, "organisation").otherwise("person / other").alias("party"))
               .agg(F.count(F.lit(1)).alias("customer_months"),
                    F.avg(((F.col("card_in") + F.col("card_out")) > 0).cast("double")).alias("use_debit_card"),
                    F.avg(F.col("expl1").cast("double")).alias("explained_without_card"),
                    F.avg(F.col("expl2").cast("double")).alias("explained_with_card"),
                    (F.sum(F.abs("resid")) / F.sum("activity")).alias("unexplained_dollars_without_card"),
                    (F.sum(F.abs("resid2")) / F.sum("activity2")).alias("unexplained_dollars_with_card"),
                    F.sum("card_out").alias("card_out"), F.sum("card_in").alias("card_in"),
                    F.sum(F.abs("resid")).alias("residual_without_card"))
               .orderBy("party").toPandas())
    CELL.show(("Debit card: how much of the gap it closes", pct_tbl(_cr, ["use_debit_card", "explained_without_card", "explained_with_card",
                                                                          "unexplained_dollars_without_card", "unexplained_dollars_with_card"])))
    for r in _cr.itertuples():
        ans(f"Debit card, {r.party}: customer-months explained", f"{r.explained_without_card} without card, {r.explained_with_card} with card")

## §6 · Daily check (sample)

For a sample of customers in one month: each business day's balance change against that day's payments (daily
edges), and against a 3-day window to absorb posting-date differences. A day with a balance move and no payment
within a day either side is an action we do not see.

In [ ]:
if DAILY_OK:
    CELL.start("§6 · Daily check (sample)")
    dd = SRC["DAILY_TABLE"]
    _dcol, _bcol = DAILY_COLS
    with step("daily balances and daily payments"):
        _dmonths = sorted(set(table_months(CONFIG["EDGE_DAILY"])) & (set(table_months(CONFIG["DAILY_TABLE"])) if "month" in dd.columns else set(REC_MONTHS)))
        DM = CONFIG["DAILY_MONTH"] or (_dmonths[-1] if _dmonths else REC_MONTHS[-1])
        keep = lambda c: F.abs(F.hash(F.col(c).cast("string"))) % 100 < CONFIG["DAILY_PCT"]
        _db = (dd.where((F.col("month") == DM) if "month" in dd.columns else (F.date_format(F.col(_dcol), "yyyy-MM") == DM))
               .where(keep(ID)).select(F.col(ID).cast("string").alias(ID), F.to_date(_dcol).alias("day"), F.col(_bcol).cast("double").alias("bal"))
               .groupBy(ID, "day").agg(F.sum("bal").alias("bal")))
        _wd = W.partitionBy(ID).orderBy("day")
        _db = _db.withColumn("d", F.col("bal") - F.lag("bal").over(_wd)).where(F.col("d").isNotNull())
        _mp = _map.where(keep(ID))
        ed = SRC["EDGE_DAILY"].where(F.col("month") == DM).where(~F.coalesce(F.col("is_self").cast("boolean"), F.lit(False)))
        _in = (ed.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").cast("string").alias("mdm_id"), F.col("txn_date").alias("day"),
                                                            F.col("amount").cast("double").alias("x_in"), F.lit(0.0).alias("x_out")))
        _out = (ed.where(F.col("src_kind") == "CUST").select(F.col("src_id").cast("string").alias("mdm_id"), F.col("txn_date").alias("day"),
                                                             F.lit(0.0).alias("x_in"), F.col("amount").cast("double").alias("x_out")))
        _fl = (_in.unionByName(_out).join(_mp, "mdm_id").groupBy(ID, F.to_date("day").alias("day"))
               .agg(F.sum("x_in").alias("f_in"), F.sum("x_out").alias("f_out")))
        J = (_db.join(_fl, [ID, "day"], "full").na.fill(0.0, ["f_in", "f_out"])
             .withColumn("net", F.col("f_in") - F.col("f_out")))
        _w3 = W.partitionBy(ID).orderBy("day").rowsBetween(-1, 1)
        J = (J.withColumn("flow3", F.sum(F.col("f_in") + F.col("f_out")).over(_w3))
             .withColumn("d3", F.sum(F.coalesce("d", F.lit(0.0))).over(_w3)).withColumn("net3", F.sum("net").over(_w3)))
        _day = J.where(F.col("d").isNotNull()).agg(
            F.countDistinct(ID).alias("customers"), F.count(F.lit(1)).alias("customer_days"),
            F.avg((F.abs("d") > CONFIG["TOL_ABS"]).cast("double")).alias("days_balance_moved"),
            F.avg(((F.abs("d") > CONFIG["TOL_ABS"]) & ((F.col("f_in") + F.col("f_out")) <= 0)).cast("double")).alias("moved_no_payment_that_day"),
            F.avg(((F.abs("d") > CONFIG["TOL_ABS"]) & (F.col("flow3") <= 0)).cast("double")).alias("moved_no_payment_within_1_day"),
            (F.sum(F.abs(F.col("d") - F.col("net"))) / F.sum(F.abs("d") + F.col("f_in") + F.col("f_out"))).alias("unexplained_share_1_day"),
            (F.sum(F.abs(F.col("d3") - F.col("net3"))) / F.sum(F.abs("d3") + F.col("flow3"))).alias("unexplained_share_3_days")).toPandas()
        _nobal = J.where(F.col("d").isNull() & ((F.col("f_in") + F.col("f_out")) > 0)).agg(
            F.count(F.lit(1)).alias("payment_days_without_a_balance_row"), F.countDistinct(ID).alias("customers")).toPandas()
    _day.insert(0, "month", DM)
    CELL.show((f"Daily: {CONFIG['DAILY_PCT']}% of customers, {DM}",
               pct_tbl(_day, ["days_balance_moved", "moved_no_payment_that_day", "moved_no_payment_within_1_day",
                              "unexplained_share_1_day", "unexplained_share_3_days"])),
              ("Days with payments but no balance row (weekends and holidays post to the next business day)", _nobal))
    ans(f"Daily ({DM}): balance moved with no payment within a day", _day.moved_no_payment_within_1_day.iloc[0])

## §7 · Answer sheet

In [ ]:
CELL.start("§7 · Answer sheet")
pd.DataFrame(ANSWERS).to_csv(os.path.join(OUT_DIR, "answers.csv"), index=False)
CELL.show(("Answers", pd.DataFrame(ANSWERS)),
          (f"Figures in {OUT_DIR}", pd.DataFrame(FIG_INDEX, columns=["file", "caption"])),
          ("Timings (minutes)", pd.DataFrame(TIMINGS, columns=["step", "minutes"])))